## 1 · Setup, download & merge NHANES 2013–2014

Imports, then download each `.xpt` file and merge on `SEQN` (one row per respondent).

In [ ]:
# ============================================================
# BoneBot - NHANES 2013-2014 training pipeline (v2, evidence-refined)
#   Lightweight : P(osteoporosis) triage   (logistic: age + BMI + menopause)
#   Heavyweight : estimated T-score         (ridge)
#   Benchmark   : does menopause + wearable activity beat the age+weight baseline?
#
# Evidence (MOTIVATION.md, sec 2): for women, prediction "collapses to age + BMI" --
# exactly what OST / the existing tools use. Our differentiators are the two modalities
# the field DROPS: menopause status + objective wearable activity. So the winning claim
# is the ABLATION (do those add value over age+BMI?), not "we beat OST outright".
# ============================================================

import ssl, urllib.request, io
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.covariance import MinCovDet
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, r2_score, roc_auc_score,
                             average_precision_score, roc_curve)

ssl._create_default_https_context = ssl._create_unverified_context

# --- 1. download + merge (all one row per person -> safe to merge on SEQN) ---
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def load(f):
    req = urllib.request.Request(BASE + f + ".xpt", headers=HEADERS)
    with urllib.request.urlopen(req) as r:
        return pd.read_sas(io.BytesIO(r.read()), format="xport")

files = ["DEMO_H", "BMX_H", "DXXFEM_H", "RHQ_H", "OSQ_H", "SMQ_H",
         "ALQ_H", "MCQ_H", "VID_H", "BIOPRO_H", "PAQ_H"]
frames = {}
for f in files:
    try:
        frames[f] = load(f)
    except Exception as e:
        print("MISSING / renamed:", f, e)

merged = frames["DEMO_H"]
for f in files[1:]:
    if f in frames:
        merged = merged.merge(frames[f], on="SEQN", how="left")

REF_MEAN, REF_SD = 0.858, 0.120   # NHANES III young-adult white-female femoral-neck reference


## 2 · Build the analytic sample & features

Women 50+ with a femur DXA. Derive the 13 model features and the T-score label; the DXA scan is the label only, never a feature.

In [ ]:
# --- 2. heavyweight sample: women 50+ with a femur DXA scan ---
women = merged[(merged.RIAGENDR == 2) & (merged.RIDAGEYR >= 50) & (merged.DXXNKBMD.notna())].copy()

women["Tscore"]                 = (women["DXXNKBMD"] - REF_MEAN) / REF_SD      # target
women["age"]                    = women["RIDAGEYR"]
women["bmi"]                    = women["BMXBMI"]
women["vitaminD"]               = women["LBXVIDMS"]                            # nmol/L (evidence-backed nutritional)
women["calcium"]                = women["LBDSCASI"]                            # mmol/L (evidence-backed nutritional)
women["currentSmoker"]          = women["SMQ040"].isin([1, 2]).astype(int)
women["priorFragilityFracture"] = women[["OSQ010A","OSQ010B","OSQ010C"]].eq(1).any(axis=1).astype(int)
women["glucocorticoids"]        = (women["OSQ130"] == 1).astype(int)
women["highAlcohol"]            = (women["ALQ130"] >= 3).astype(int)
women["yearsSinceMenopause"]    = (women["RIDAGEYR"] - women["RHQ060"]).clip(lower=0)
# secondary osteoporosis: thyroid disease (MCQ160M, in MCQ_H) OR chronic kidney
# disease (eGFR < 60 from serum creatinine LBXSCR, in BIOPRO_H, via CKD-EPI 2021,
# race-free, female). Coeliac is NOT in 2013-2014 (serology ran only 2009-2010),
# so it is excluded from the trained flag. One binary: 1 if either, else 0.
scr = women["LBXSCR"]  # serum creatinine, mg/dL
k, alpha = 0.7, -0.241
egfr = (142 * np.minimum(scr / k, 1) ** alpha
        * np.maximum(scr / k, 1) ** -1.200
        * 0.9938 ** women["RIDAGEYR"] * 1.012)
women["thyroid"] = (women["MCQ160M"] == 1)
women["ckd"]     = (egfr < 60)
women["secondaryCondition"] = (women["thyroid"] | women["ckd"]).astype(int)
# rheumatoid arthritis: arthritis ever (MCQ160A) AND type = rheumatoid (MCQ195==2)
women["rheumatoidArthritis"] = ((women["MCQ160A"] == 1) & (women["MCQ195"] == 2)).astype(int)
# hormone therapy: ever used female hormones (RHQ540)
women["onHormoneTherapy"] = (women["RHQ540"] == 1).astype(int)
# (dropped rbc + alp: no supporting evidence in MOTIVATION.md -> they were noise)

# wearable activity: mean daily MIMS from the wrist accelerometer, normalised 0-1
pax = load("PAXDAY_H")
# Valid wear day: >= 10 h (600 min) of wake-wear time (PAXWWMD) — the standard
# NHANES accelerometry criterion. Average MIMS only over valid days, and require
# >= 4 of them, so we never conflate "did not wear the monitor" with "did not
# move". Persons below that get NaN activity and are median-imputed downstream,
# like any other missing feature. (PAXQFD is a flagged-minute count, not a 0/1
# quality flag, so it is not used as a day filter.)
pax_valid = pax[pax["PAXWWMD"] >= 600]
day_stats = pax_valid.groupby("SEQN")["PAXMTSD"].agg(["mean", "size"]).reset_index()
day_stats = day_stats[day_stats["size"] >= 4]
act = day_stats[["SEQN", "mean"]].rename(columns={"mean": "daily_mims"})
women = women.merge(act, on="SEQN", how="left")                                 # aggregated first -> no dup rows
women["activity_level"] = (women["daily_mims"] / women["daily_mims"].quantile(0.95)).clip(0, 1)

print("heavyweight sample:", women.shape[0], "women")


## 3 · Lightweight triage model (age + BMI + menopause)

Logistic P(osteoporosis) used to decide whether the fuller questionnaire is worth the user's time.

In [ ]:
# --- 3. lightweight triage: P(osteoporosis) from age + BMI + menopause (BROAD age range) ---
# BMI added: the evidence says weight/BMI is the #1 co-predictor with age (that's what OST
# uses). Without it the triage cannot match OST.
fem = merged[(merged.RIAGENDR == 2) & (merged.DXXNKBMD.notna()) & (merged.RIDAGEYR >= 18)].copy()
fem["osteoporosis"]   = ((fem["DXXNKBMD"] - REF_MEAN) / REF_SD <= -2.5).astype(int)
fem["age"]            = fem["RIDAGEYR"]
fem["bmi"]            = fem["BMXBMI"]
fem["postmenopausal"] = ((fem["RIDAGEYR"] - fem["RHQ060"]) >= 0).astype(int)

triage_feats = ["age", "bmi", "postmenopausal"]
Xl = fem[triage_feats].apply(lambda c: c.fillna(c.median()))
yl = fem["osteoporosis"]
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(Xl, yl, test_size=0.25, random_state=0, stratify=yl)
triage = LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l)
probs_l = triage.predict_proba(Xte_l)[:, 1]
triage_auc = roc_auc_score(yte_l, probs_l)

def prob_osteoporosis(age, bmi, postmenopausal):
    row = pd.DataFrame([[age, bmi, postmenopausal]], columns=triage_feats)
    return triage.predict_proba(row)[0, 1]        # column 1 = P(osteoporosis)

# triage cutoff: keep high sensitivity like the existing tools (~89-94%).
# Refer everyone at/above the cutoff; safely exclude the confident-negatives below it.
fpr, tpr, thr = roc_curve(yte_l, probs_l)
target_sens = 0.95
i = np.where(tpr >= target_sens)[0][0]
cutoff = float(thr[i])
excluded = float((probs_l < cutoff).mean())


## 4 · Heavyweight T-score model — split → impute → outlier-filter → fit → evaluate

Split BEFORE any preprocessing (no leakage): training-only median imputation, MinCovDet outlier filter, standardised Ridge, then held-out MAE / R².

In [ ]:
# --- 4. heavyweight: estimated T-score (scaled ridge) ---
# Evidence-backed set: age + BMI (core, like the tools) + our differentiators
# (menopause, wearable activity) + FRAX factors + nutritional (vit D, calcium).
features = ["age", "bmi", "yearsSinceMenopause", "activity_level",
            "priorFragilityFracture", "glucocorticoids", "currentSmoker",
            "highAlcohol", "vitaminD", "calcium", "rheumatoidArthritis",
            "onHormoneTherapy", "secondaryCondition"]

continuous_features = ["age", "bmi", "yearsSinceMenopause",
                       "activity_level", "vitaminD", "calcium"]

# 4a. Split BEFORE imputation, outlier detection, or scaling. This keeps all
# information from the held-out test set out of the training procedure.
X_raw = women[features].copy()
y = women["Tscore"].copy()
Xtr_raw, Xte_raw, ytr_raw, yte_h = train_test_split(
    X_raw, y, test_size=0.25, random_state=0
)

# 4b. Learn medians from the training set only, then use those same values
# for both training and test data. This avoids preprocessing leakage.
training_medians = Xtr_raw.median()
Xtr_imputed = Xtr_raw.fillna(training_medians)
Xte_imputed = Xte_raw.fillna(training_medians)

# 4c. Detect multivariate outliers using continuous predictors only. Robust
# covariance reduces distortion from the very observations being assessed.
# The detector is fitted on training data only; the test set remains intact.
OUTLIER_QUANTILE = 0.99
robust_covariance = MinCovDet(random_state=0).fit(
    Xtr_imputed[continuous_features]
)
mahalanobis_squared = robust_covariance.mahalanobis(
    Xtr_imputed[continuous_features]
)
mahalanobis_cutoff = chi2.ppf(
    OUTLIER_QUANTILE, df=len(continuous_features)
)
retain_training_row = mahalanobis_squared <= mahalanobis_cutoff
Xtr_filtered = Xtr_imputed.loc[retain_training_row].copy()
ytr_h = ytr_raw.loc[Xtr_filtered.index].copy()
outliers_removed = int((~retain_training_row).sum())

# 4d. Standardise every predictor using the retained training rows only.
# Ridge is scale-sensitive, so comparable scales make its penalty consistent.
scaler = StandardScaler()
Xtr_h = pd.DataFrame(
    scaler.fit_transform(Xtr_filtered),
    columns=features, index=Xtr_filtered.index,
)
Xte_h = pd.DataFrame(
    scaler.transform(Xte_imputed),
    columns=features, index=Xte_imputed.index,
)

# 4e. Keep the original modelling technique and alpha: Ridge(alpha=1.0).
model = Ridge(alpha=1.0).fit(Xtr_h, ytr_h)
pred_h = model.predict(Xte_h)

# Convert standardised coefficients back to raw input units. These values
# can be compared fairly with the previous model and used by the current app.
raw_scale_coefficients = model.coef_ / scaler.scale_
raw_scale_intercept = model.intercept_ - np.sum(
    model.coef_ * scaler.mean_ / scaler.scale_
)

perf_table = pd.DataFrame({
    "Metric": ["MAE", "R2", "n_train_retained", "n_test_untouched", "training_outliers_removed"],
    "Value":  [round(mean_absolute_error(yte_h, pred_h), 3),
               round(r2_score(yte_h, pred_h), 3), len(Xtr_h), len(Xte_h), outliers_removed],
})
coef_table = pd.DataFrame({
    "Term": ["intercept"] + features,
    "Standardised coefficient": [model.intercept_] + list(model.coef_),
    "Raw-scale equivalent": [raw_scale_intercept] + list(raw_scale_coefficients),
})
coef_table["Direction"] = coef_table["Standardised coefficient"].apply(
    lambda c: "raises T-score" if c > 0 else "lowers T-score")
coef_table = coef_table.sort_values("Standardised coefficient").reset_index(drop=True)


## 5 · Benchmark ablation

Does menopause + wearable activity beat the age + BMI baseline the existing tools use?

In [ ]:
# --- 5. benchmark ABLATION: does menopause + wearable activity beat age+weight? ---
# The honest scientific claim (MOTIVATION sec 3a/4d): the field drops menopause + activity;
# we show they ADD value over the age+BMI baseline the existing tools already use.
women["osteoporosis"] = (women["Tscore"] <= -2.5).astype(int)
yb_tr = women.loc[Xtr_h.index, "osteoporosis"]
yb_te = women.loc[Xte_h.index, "osteoporosis"]

ablation = {
    "baseline (age + BMI) ~ current tools": ["age", "bmi"],
    "+ menopause":                          ["age", "bmi", "yearsSinceMenopause"],
    "+ wearable activity":                  ["age", "bmi", "yearsSinceMenopause", "activity_level"],
    "full model":                           features,
}
print("\n=== Ablation: does our added signal beat the age+weight baseline? (osteoporosis classifier, same split) ===")
for name, feats in ablation.items():
    Xb = women[feats].apply(lambda c: c.fillna(c.median()))
    clf = LogisticRegression(max_iter=1000).fit(Xb.loc[Xtr_h.index], yb_tr)
    prob = clf.predict_proba(Xb.loc[Xte_h.index])[:, 1]
    print(f"   {name:38s}  AUC {roc_auc_score(yb_te, prob):.3f}   PR-AUC {average_precision_score(yb_te, prob):.3f}")

ost = women.loc[Xte_h.index].copy()
ost["ost"] = 0.2 * (ost["BMXWT"] - ost["age"])
mm = ost["ost"].notna()
print(f"   {'OST tool (age + weight, reference)':38s}  AUC {roc_auc_score(ost.loc[mm,'osteoporosis'], -ost.loc[mm,'ost']):.3f}")


## 6 · Results

Held-out metrics and coefficient table.

In [1]:
# --- 6. results ---
print(f"\nLIGHTWEIGHT triage (age + BMI + menopause) - AUC: {triage_auc:.3f}")
print(f"Cutoff for {target_sens:.0%} sensitivity: refer if P(osteoporosis) >= {cutoff:.3f}  "
      f"(safely excludes {excluded:.0%} of women as low-risk)")
for a, b, mmeno in [(23, 25, 0), (43, 25, 0), (55, 24, 1), (70, 22, 1)]:
    print(f"   age {a}, bmi {b}, postmeno {mmeno}: P(osteoporosis) = {prob_osteoporosis(a, b, mmeno):.1%}")

print("\nHEAVYWEIGHT T-score model")
print(perf_table.to_string(index=False))
print()
print(coef_table.to_string(index=False))

heavyweight sample: 1119 women

=== Ablation: does our added signal beat the age+weight baseline? (osteoporosis classifier, same split) ===
   baseline (age + BMI) ~ current tools    AUC 0.772   PR-AUC 0.280
   + menopause                             AUC 0.774   PR-AUC 0.286
   + wearable activity                     AUC 0.773   PR-AUC 0.301
   full model                              AUC 0.789   PR-AUC 0.350
   OST tool (age + weight, reference)      AUC 0.779

LIGHTWEIGHT triage (age + BMI + menopause) - AUC: 0.867
Cutoff for 95% sensitivity: refer if P(osteoporosis) >= 0.029  (safely excludes 49% of women as low-risk)
   age 23, bmi 25, postmeno 0: P(osteoporosis) = 0.3%
   age 43, bmi 25, postmeno 0: P(osteoporosis) = 1.8%
   age 55, bmi 24, postmeno 1: P(osteoporosis) = 5.0%
   age 70, bmi 22, postmeno 1: P(osteoporosis) = 20.3%

HEAVYWEIGHT T-score model
                   Metric   Value
                      MAE   0.727
                       R2   0.276
         n_train_retained 

## 7 · Triage threshold protocol

In [ ]:
# --- 7. Lightweight triage threshold protocol ---
# Threshold selection uses validation data only. The held-out test set is used once,
# after the threshold is locked, for a retrospective research-screening audit.
# Locate the helper whether this notebook is run from the repository root,
# from model/, or from this Desktop working copy.
from pathlib import Path
import sys

triage_helper_locations = [
    Path.cwd(),
    Path.cwd() / "model",
]
triage_helper_dir = next(
    (path for path in triage_helper_locations if (path / "triage_audit.py").exists()),
    None,
)
if triage_helper_dir is None:
    raise FileNotFoundError("Could not locate model/triage_audit.py")
if str(triage_helper_dir) not in sys.path:
    sys.path.insert(0, str(triage_helper_dir))

from triage_audit import audit_threshold, select_threshold

X_train, X_temp, y_train, y_temp = train_test_split(
    Xl, yl, test_size=0.40, random_state=17, stratify=yl
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=17, stratify=y_temp
)

locked_triage = LogisticRegression(max_iter=1000).fit(X_train, y_train)
valid_probabilities = locked_triage.predict_proba(X_valid)[:, 1]
test_probabilities = locked_triage.predict_proba(X_test)[:, 1]

# Safety-first routing: require >=99% sensitivity on validation so the triage
# misses ~1% of DXA-defined cases, not ~5%. select_threshold returns the HIGHEST
# candidate meeting the target, so tightening 0.95 -> 0.99 moves the locked cut
# from 2% to 1% (the 1% row is the only one clearing 99% sensitivity). The cost
# is more referrals to the full assessment; the benefit is fewer missed cases.
# Keep this in sync with triage.threshold in model/model-parameters.ts.
TARGET_SENSITIVITY = 0.99
CANDIDATE_THRESHOLDS = np.round(np.arange(0.01, 0.101, 0.01), 2)
validation_rows = []
for candidate in CANDIDATE_THRESHOLDS:
    candidate_audit = audit_threshold(y_valid, valid_probabilities, candidate, bootstrap_samples=0)
    validation_rows.append({
        "threshold": candidate,
        "sensitivity": candidate_audit["sensitivity"],
        "NPV": candidate_audit["npv"],
        "false negatives": candidate_audit["false_negatives"],
    })
validation_table = pd.DataFrame(validation_rows)
print("=== Validation-only threshold comparison (do not use test results here) ===")
print(validation_table.to_string(index=False, formatters={"sensitivity": "{:.1%}".format, "NPV": "{:.1%}".format}))

try:
    FINAL_TRIAGE_THRESHOLD = select_threshold(
        y_valid, valid_probabilities, CANDIDATE_THRESHOLDS, TARGET_SENSITIVITY
    )
except ValueError as error:
    raise RuntimeError("No candidate met the pre-specified validation safety target; do not deploy a routing threshold.") from error

final_audit = audit_threshold(
    y_test, test_probabilities, FINAL_TRIAGE_THRESHOLD, bootstrap_samples=2000, random_state=17
)
print(f"\nLocked triage threshold: {FINAL_TRIAGE_THRESHOLD:.0%} (selected on validation for ≥{TARGET_SENSITIVITY:.0%} sensitivity)")
print("=== Final held-out test audit at the locked threshold ===")
print(f"Sensitivity: {final_audit['sensitivity']:.1%} (95% CI {final_audit['sensitivity_ci95'][0]:.1%}–{final_audit['sensitivity_ci95'][1]:.1%})")
print(f"NPV: {final_audit['npv']:.1%} (95% CI {final_audit['npv_ci95'][0]:.1%}–{final_audit['npv_ci95'][1]:.1%})")
print(f"False negatives: {final_audit['false_negatives']} of {(y_test == 1).sum()} DXA-defined cases")
print(f"Brier score: {final_audit['brier_score']:.3f} (lower is better)")
print("Calibration bins (predicted risk vs observed prevalence):")
print(pd.DataFrame(final_audit["calibration_bins"]).query("count > 0").to_string(index=False))
print("Research screening only: this audit does not diagnose osteoporosis or establish a clinical referral rule.")

## 8 · Export coefficients → model-parameters.ts

In [3]:
# --- 7. EXPORT -> src/lib/../model/model-parameters.ts -----------------------
# Run after the scaled ridge fit. Produces the raw-scale numbers needed by
# model-parameters.ts plus the fitted scaler values for reproducibility:
#   - standardised coefficients and raw-scale equivalents
#   - featureVariances + residualStd for the per-person prediction interval
# The interval widens per imputed feature by coef^2 * Var(feature); residualStd
# is the complete-data residual SD (band = z * sqrt(residualStd^2 + sum of those)).
import json
import numpy as np

# TS coefficient keys differ from the pandas names for activity only.
TS_KEY = {"activity_level": "activityLevel"}
def ts(name): return TS_KEY.get(name, name)

resid = np.asarray(yte_h) - np.asarray(pred_h)
residual_std = float(np.std(resid, ddof=1))

# Variance of each feature over observed raw values. Raw-scale coefficients
# preserve the current app formula even though fitting used scaled inputs.
feat_var = {ts(f): round(float(women[f].dropna().var(ddof=1)), 6) for f in features}
standardised_coef = {ts(f): round(float(model.coef_[i]), 6) for i, f in enumerate(features)}
coef = {ts(f): round(float(raw_scale_coefficients[i]), 6) for i, f in enumerate(features)}
scaler_means = {ts(f): round(float(scaler.mean_[i]), 6) for i, f in enumerate(features)}
scaler_scales = {ts(f): round(float(scaler.scale_[i]), 6) for i, f in enumerate(features)}

print("raw-scale intercept for app:", round(float(raw_scale_intercept), 6))
print("\nstandardised coefficients:\n" + json.dumps(standardised_coef, indent=2))
print("\nraw-scale equivalent coefficients for app:\n" + json.dumps(coef, indent=2))
print("\nscaler means:\n" + json.dumps(scaler_means, indent=2))
print("\nscaler scales:\n" + json.dumps(scaler_scales, indent=2))
print("\nintervals.residualStd:", round(residual_std, 4))
print("intervals.z: 1.96   # 95% normal")
print("\nintervals.featureVariances:\n" + json.dumps(feat_var, indent=2))
print("\nsecondaryCondition coefficient:", coef.get("secondaryCondition"))
print("secondaryCondition prevalence:", round(float(women["secondaryCondition"].mean()), 4))
imp = {ts(f): round(float(training_medians[f]), 4) for f in features}
print("\nimputationDefaults (training medians):\n" + json.dumps(imp, indent=2))
print("\nNOTE: paste featureVariances for every feature the deployed model uses.")
print("If RA / HRT stay in model-parameters, add their variances too,")
print("or drop those coefficients so the model and notebook match.")

# Coverage check: does the adaptive 95% band actually cover ~95% under missingness?
# Re-impute a random subset of columns per test row with the train medians and
# confirm empirical coverage tracks the nominal 0.95.
rng = np.random.default_rng(0)
med = training_medians
z = 1.96
covered, widths = [], []
Xte = Xte_imputed.copy()
for idx in Xte.index:
    row = Xte.loc[idx].copy()
    keep = rng.random(len(features)) < 0.5   # ~half the inputs supplied
    var = residual_std ** 2
    for j, f in enumerate(features):
        if not keep[j]:
            row[f] = med[f]
            var += float(raw_scale_coefficients[j]) ** 2 * float(women[f].dropna().var(ddof=1))
    scaled_row = pd.DataFrame(
        scaler.transform(row.to_frame().T), columns=features, index=[idx]
    )
    pred = float(model.predict(scaled_row)[0])
    hw = z * np.sqrt(var)
    covered.append(abs(float(y.loc[idx]) - pred) <= hw)
    widths.append(hw)
print(f"\nadaptive-band empirical coverage under ~50% missingness: {np.mean(covered):.3f} (target 0.95)")
print(f"median half-width: {np.median(widths):.3f}  (fixed was {0.9467})")


raw-scale intercept for app: -4.024468

standardised coefficients:
{
  "age": -0.294864,
  "bmi": 0.443152,
  "yearsSinceMenopause": 0.019576,
  "activityLevel": 0.04249,
  "priorFragilityFracture": -0.114922,
  "glucocorticoids": -0.026439,
  "currentSmoker": -1.3e-05,
  "highAlcohol": -0.008763,
  "vitaminD": -0.022483,
  "calcium": 0.092809,
  "rheumatoidArthritis": -0.000486,
  "onHormoneTherapy": 0.062031,
  "secondaryCondition": -0.014877
}

raw-scale equivalent coefficients for app:
{
  "age": -0.03187,
  "bmi": 0.069652,
  "yearsSinceMenopause": 0.001773,
  "activityLevel": 0.279435,
  "priorFragilityFracture": -0.36487,
  "glucocorticoids": -0.103762,
  "currentSmoker": -3.6e-05,
  "highAlcohol": -0.033828,
  "vitaminD": -0.000731,
  "calcium": 1.124829,
  "rheumatoidArthritis": -0.001976,
  "onHormoneTherapy": 0.130588,
  "secondaryCondition": -0.03161
}

scaler means:
{
  "age": 64.177665,
  "bmi": 28.636294,
  "yearsSinceMenopause": 18.143401,
  "activityLevel": 0.691211,
 

## 9 · Multicollinearity & coefficient-stability audit

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

BOOTSTRAPS = 1000
VIF_LIMIT = 5
RANDOM_SEED = 42

# --- 8. MULTICOLLINEARITY AND COEFFICIENT-STABILITY AUDIT ---
# VIF is calculated on the final scaled training matrix. The bootstrap below
# refits the scaler inside every sample before refitting Ridge(alpha=1.0).
X_audit = Xtr_h.copy()
y_audit = ytr_h.copy()

# Calculate VIF for every predictor.
vif_values = {}

for variable in features:
    other_variables = X_audit.drop(columns=[variable])

    auxiliary_model = LinearRegression()
    auxiliary_model.fit(other_variables, X_audit[variable])

    r_squared = auxiliary_model.score(
        other_variables,
        X_audit[variable],
    )

    vif_values[variable] = (
        np.inf if r_squared >= 1 else 1 / (1 - r_squared)
    )

# Bootstrap the scaled Ridge model. Resampling the retained raw-scale rows
# and refitting StandardScaler each time captures preprocessing uncertainty.
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_coefficients = {
    variable: [] for variable in features
}

for _ in range(BOOTSTRAPS):
    sampled_positions = rng.integers(
        low=0,
        high=len(Xtr_filtered),
        size=len(Xtr_filtered),
    )

    X_sample_raw = Xtr_filtered.iloc[sampled_positions]
    y_sample = y_audit.iloc[sampled_positions]

    bootstrap_scaler = StandardScaler()
    X_sample = bootstrap_scaler.fit_transform(X_sample_raw)
    bootstrap_model = Ridge(alpha=1.0)
    bootstrap_model.fit(X_sample, y_sample)

    for variable, coefficient in zip(
        features,
        bootstrap_model.coef_,
    ):
        bootstrap_coefficients[variable].append(coefficient)

# Fit the old unscaled/no-outlier model once for a fair raw-unit comparison.
# Its split and alpha match the earlier notebook implementation.
X_previous = women[features].apply(lambda c: c.fillna(c.median()))
Xtr_previous, _, ytr_previous, _ = train_test_split(
    X_previous, y, test_size=0.25, random_state=0
)
previous_model = Ridge(alpha=1.0).fit(Xtr_previous, ytr_previous)
previous_coefficients = dict(zip(features, previous_model.coef_))

# Build the PASS/FAIL table for every predictor. A VIF below 5 passes. A
# bootstrap interval passes only when its 95% interval excludes zero.
rows = []

for variable in features:
    coefficients = np.asarray(
        bootstrap_coefficients[variable]
    )

    median_coefficient = np.median(coefficients)
    interval_low, interval_high = np.quantile(
        coefficients,
        [0.025, 0.975],
    )

    vif = vif_values[variable]

    rows.append({
        "Variable": variable,
        "VIF": vif,
        "VIF test": "PASS" if vif < VIF_LIMIT else "FAIL",
        "Previous raw coefficient": previous_coefficients[variable],
        "Fitted standardised coefficient": model.coef_[features.index(variable)],
        "New raw-scale equivalent": raw_scale_coefficients[features.index(variable)],
        "Bootstrap median (standardised)": median_coefficient,
        "95% interval lower": interval_low,
        "95% interval upper": interval_high,
        "Interval test": (
            "PASS"
            if interval_low > 0 or interval_high < 0
            else "FAIL"
        ),
    })

audit_table = pd.DataFrame(rows)

display(
    audit_table.style
    .format({
        "VIF": "{:.2f}",
        "Previous raw coefficient": "{:.4f}",
        "Fitted standardised coefficient": "{:.4f}",
        "New raw-scale equivalent": "{:.4f}",
        "Bootstrap median (standardised)": "{:.4f}",
        "95% interval lower": "{:.4f}",
        "95% interval upper": "{:.4f}",
    })
    .map(
        lambda value: (
            "background-color: #d9ead3"
            if value == "PASS"
            else "background-color: #f4cccc"
            if value == "FAIL"
            else ""
        )
    )
)

,Variable,VIF,VIF test,Previous raw coefficient,Fitted standardised coefficient,New raw-scale equivalent,Bootstrap median (standardised),95% interval lower,95% interval upper,Interval test
0,age,2.17,PASS,-0.0318,-0.2949,-0.0319,-0.2944,-0.3845,-0.1980,PASS
1,bmi,1.17,PASS,0.0663,0.4432,0.0697,0.4448,0.3690,0.5200,PASS
2,yearsSinceMenopause,2.01,PASS,0.0020,0.0196,0.0018,0.0201,-0.0662,0.1047,FAIL
3,activity_level,1.22,PASS,0.3208,0.0425,0.2794,0.0415,-0.0371,0.1193,FAIL
4,priorFragilityFracture,1.07,PASS,-0.3772,-0.1149,-0.3649,-0.1129,-0.1808,-0.0473,PASS
5,glucocorticoids,1.03,PASS,-0.1353,-0.0264,-0.1038,-0.0283,-0.0956,0.0451,FAIL
6,currentSmoker,1.16,PASS,0.0088,-0.0000,-0.0000,-0.0009,-0.0822,0.0774,FAIL
7,highAlcohol,1.10,PASS,-0.0693,-0.0088,-0.0338,-0.0075,-0.0731,0.0573,FAIL
8,vitaminD,1.20,PASS,-0.0011,-0.0225,-0.0007,-0.0227,-0.0942,0.0505,FAIL
9,calcium,1.07,PASS,0.9759,0.0928,1.1248,0.0928,0.0272,0.1563,PASS
